In [1]:
"""
================================================================================
NCAA March Mania 2026 — FINAL SUBMISSION GENERATOR
================================================================================
Produces: shaurya_final_3.csv  (132,133 rows)
Men   (TeamID 1xxx) : trained on M files, men's best BO params
Women (TeamID 3xxx) : trained on W files, women's best BO params

Steps:
  1  Load data
  2  Compute ELO (2010–2026 regular season)
  3  Build team stats, recent form, Massey (men only), seeds
  4  Build training matchup features
  5  Train LightGBM + XGBoost, tune ensemble weight
  6  Build 2026 prediction features (2026 regular season stats + end-of-2026 ELO)
  7  Predict all rows → clip to [0.025, 0.975]
  8  Save + sanity check
================================================================================
"""

import pandas as pd
import numpy as np
import warnings
import os
from itertools import combinations

warnings.filterwarnings("ignore")

# ── Try importing ML libs gracefully ──────────────────────────────────────────
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("⚠  lightgbm not found – will use XGBoost only")

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠  xgboost not found – will use LightGBM only")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

assert HAS_LGB or HAS_XGB, "Need at least one of lightgbm or xgboost installed."

# ─────────────────────────────────────────────────────────────────────────────
# 0.  DATA PATH  ── change this one variable to wherever your CSVs live
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR = "/Users/shaurya/Downloads"   # ← adjust if needed
OUT_NAME  = "shaurya_final_3.csv"

def p(fname):
    return os.path.join(DATA_DIR, fname)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Load Data
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("STEP 1 — Loading data")
print("="*70)

# Teams
m_teams  = pd.read_csv(p("MTeams.csv"))
w_teams  = pd.read_csv(p("WTeams.csv"))

# Seeds
m_seeds  = pd.read_csv(p("MNCAATourneySeeds.csv"))
w_seeds  = pd.read_csv(p("WNCAATourneySeeds.csv"))

# Regular season results
m_reg_compact   = pd.read_csv(p("MRegularSeasonCompactResults.csv"))
w_reg_compact   = pd.read_csv(p("WRegularSeasonCompactResults.csv"))
m_reg_detailed  = pd.read_csv(p("MRegularSeasonDetailedResults.csv"))
w_reg_detailed  = pd.read_csv(p("WRegularSeasonDetailedResults.csv"))

# Tournament results (for training labels)
m_tourney = pd.read_csv(p("MNCAATourneyCompactResults.csv"))
w_tourney = pd.read_csv(p("WNCAATourneyCompactResults.csv"))

# Massey ordinals (men only)
m_ordinal = pd.read_csv(p("MMasseyOrdinals.csv"))

# Submission template
sample_sub = pd.read_csv(p("SampleSubmissionStage2.csv"))

print(f"  Submission rows : {len(sample_sub):,}")
print(f"  M tourney games : {len(m_tourney):,}")
print(f"  W tourney games : {len(w_tourney):,}")
print(f"  M reg detailed  : {len(m_reg_detailed):,}  (seasons: {m_reg_detailed.Season.min()}–{m_reg_detailed.Season.max()})")
print(f"  W reg detailed  : {len(w_reg_detailed):,}  (seasons: {w_reg_detailed.Season.min()}–{w_reg_detailed.Season.max()})")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Compute ELO  (2010 → 2026 regular season)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("STEP 2 — Computing ELO ratings (2010–2026)")
print("="*70)

def compute_elo(compact_df, k=20, initial=1500, start_season=2010):
    """
    Walk through every game chronologically and update ELO.
    Returns dict: {(season, team_id): elo_after_last_game_that_season}
    """
    df = compact_df[compact_df.Season >= start_season].copy()
    df = df.sort_values(["Season", "DayNum"]).reset_index(drop=True)

    elo = {}          # team → current ELO
    season_end = {}   # (season, team) → ELO at end of that season

    def get_elo(tid):
        return elo.get(tid, initial)

    prev_season = None
    for _, row in df.iterrows():
        s   = row.Season
        w   = row.WTeamID
        l   = row.LTeamID

        # At new season start, regress all ELOs toward mean
        if s != prev_season:
            for tid in list(elo.keys()):
                elo[tid] = 0.75 * elo[tid] + 0.25 * initial
            prev_season = s

        ew = get_elo(w)
        el = get_elo(l)

        exp_w = 1 / (1 + 10 ** ((el - ew) / 400))
        elo[w] = ew + k * (1 - exp_w)
        elo[l] = el + k * (0 - (1 - exp_w))

        season_end[(s, w)] = elo[w]
        season_end[(s, l)] = elo[l]

    return elo, season_end

m_elo_current, m_elo_history = compute_elo(m_reg_compact)
w_elo_current, w_elo_history = compute_elo(w_reg_compact)

print(f"  M unique teams with ELO : {len(m_elo_current)}")
print(f"  W unique teams with ELO : {len(w_elo_current)}")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Team Stats, Recent Form, Massey, Seeds
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("STEP 3 — Building team stats / recent form / Massey / seeds")
print("="*70)

def build_team_stats(detailed_df, compact_df, season):
    """
    Compute per-team season averages from detailed results.
    Returns DataFrame indexed by TeamID.
    """
    d = detailed_df[detailed_df.Season == season].copy()
    if len(d) == 0:
        # Fall back to compact if no detailed data for this season
        d = None

    c = compact_df[compact_df.Season == season].copy()

    rows = []

    all_teams = set(c.WTeamID) | set(c.LTeamID)

    for tid in all_teams:
        w_mask = c.WTeamID == tid
        l_mask = c.LTeamID == tid

        wins   = w_mask.sum()
        losses = l_mask.sum()
        games  = wins + losses
        win_pct = wins / games if games > 0 else 0.5

        pts_for  = c.loc[w_mask, "WScore"].sum() + c.loc[l_mask, "LScore"].sum()
        pts_ag   = c.loc[w_mask, "LScore"].sum() + c.loc[l_mask, "WScore"].sum()
        avg_pts  = pts_for / games if games > 0 else 0
        avg_pts_ag = pts_ag / games if games > 0 else 0
        point_diff = (pts_for - pts_ag) / games if games > 0 else 0

        row = {
            "TeamID"   : tid,
            "wins"     : wins,
            "losses"   : losses,
            "win_pct"  : win_pct,
            "avg_pts"  : avg_pts,
            "avg_pts_ag": avg_pts_ag,
            "point_diff": point_diff,
        }

        # Detailed stats
        if d is not None:
            dw = d[d.WTeamID == tid]
            dl = d[d.LTeamID == tid]
            dg = len(dw) + len(dl)

            if dg > 0:
                fgm  = (dw.WFGM.sum()  + dl.LFGM.sum())  / dg
                fga  = (dw.WFGA.sum()  + dl.LFGA.sum())  / dg
                fg3m = (dw.WFGM3.sum() + dl.LFGM3.sum()) / dg
                fg3a = (dw.WFGA3.sum() + dl.LFGA3.sum()) / dg
                ftm  = (dw.WFTM.sum()  + dl.LFTM.sum())  / dg
                fta  = (dw.WFTA.sum()  + dl.LFTA.sum())  / dg
                or_  = (dw.WOR.sum()   + dl.LOR.sum())   / dg
                dr_  = (dw.WDR.sum()   + dl.LDR.sum())   / dg
                ast  = (dw.WAst.sum()  + dl.LAst.sum())  / dg
                to_  = (dw.WTO.sum()   + dl.LTO.sum())   / dg
                stl  = (dw.WStl.sum()  + dl.LStl.sum())  / dg
                blk  = (dw.WBlk.sum()  + dl.LBlk.sum())  / dg
                pf   = (dw.WPF.sum()   + dl.LPF.sum())   / dg

                row.update({
                    "fg_pct" : fgm / fga if fga > 0 else 0,
                    "fg3_pct": fg3m / fg3a if fg3a > 0 else 0,
                    "ft_pct" : ftm / fta if fta > 0 else 0,
                    "avg_or" : or_,
                    "avg_dr" : dr_,
                    "avg_ast": ast,
                    "avg_to" : to_,
                    "avg_stl": stl,
                    "avg_blk": blk,
                    "avg_pf" : pf,
                    # Efficiency proxies
                    "off_eff": avg_pts / (fga + 0.44 * fta + to_ + 1e-9),
                    "def_eff": avg_pts_ag / (fga + 0.44 * fta + to_ + 1e-9),
                })
            else:
                for col in ["fg_pct","fg3_pct","ft_pct","avg_or","avg_dr",
                            "avg_ast","avg_to","avg_stl","avg_blk","avg_pf",
                            "off_eff","def_eff"]:
                    row[col] = 0.0
        else:
            for col in ["fg_pct","fg3_pct","ft_pct","avg_or","avg_dr",
                        "avg_ast","avg_to","avg_stl","avg_blk","avg_pf",
                        "off_eff","def_eff"]:
                row[col] = 0.0

        rows.append(row)

    return pd.DataFrame(rows).set_index("TeamID")


def build_recent_form(compact_df, season, last_n=10):
    """Win rate over last N regular season games."""
    d = compact_df[compact_df.Season == season].sort_values("DayNum")
    form = {}
    all_teams = set(d.WTeamID) | set(d.LTeamID)
    for tid in all_teams:
        games = d[(d.WTeamID == tid) | (d.LTeamID == tid)].tail(last_n)
        wins = (games.WTeamID == tid).sum()
        form[tid] = wins / len(games) if len(games) > 0 else 0.5
    return form


def parse_seed(seed_str):
    """'W01a' → 1,  'X16b' → 16"""
    import re
    m = re.search(r'\d+', str(seed_str))
    return int(m.group()) if m else 16


def build_seed_dict(seeds_df, season):
    d = seeds_df[seeds_df.Season == season]
    return {row.TeamID: parse_seed(row.Seed) for _, row in d.iterrows()}


def build_massey(ordinals_df, season, system="SAG", day_cutoff=133):
    """
    Extract Massey/SAG ranking for end-of-regular-season.
    Returns dict {TeamID: rank}  (lower = better)
    """
    d = ordinals_df[
        (ordinals_df.Season == season) &
        (ordinals_df.RankingDayNum <= day_cutoff)
    ]
    if "SystemName" in d.columns:
        d_sys = d[d.SystemName == system]
        if len(d_sys) == 0:
            d_sys = d   # fall back to all systems
    else:
        d_sys = d

    # Keep last ranking per team
    latest = (
        d_sys.sort_values("RankingDayNum")
             .groupby("TeamID")
             .last()
             .reset_index()
    )
    rank_col = "OrdinalRank" if "OrdinalRank" in latest.columns else latest.columns[-1]
    return dict(zip(latest.TeamID, latest[rank_col]))


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Build Training Matchup Features
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("STEP 4 — Building training matchup features")
print("="*70)

def build_matchup_features(tourney_df, reg_compact, reg_detailed,
                            seeds_df, elo_history, ordinals_df=None,
                            train_seasons=range(2010, 2026)):
    """
    For every tournament game in train_seasons, build a feature vector.
    Label = 1 if team_a wins (team_a always has lower ID → canonical ordering).
    """
    records = []

    for season in train_seasons:
        games = tourney_df[tourney_df.Season == season]
        if len(games) == 0:
            continue

        stats    = build_team_stats(reg_detailed, reg_compact, season)
        form     = build_recent_form(reg_compact, season)
        seeds    = build_seed_dict(seeds_df, season)
        massey   = build_massey(ordinals_df, season) if ordinals_df is not None else {}

        for _, row in games.iterrows():
            w, l = row.WTeamID, row.LTeamID
            # Canonical: a = lower ID
            a, b = (w, l) if w < l else (l, w)
            label = 1 if w < l else 0  # did 'a' win?

            feat = make_features(
                a, b, season, stats, form, seeds, elo_history, massey
            )
            feat["label"] = label
            records.append(feat)

    df = pd.DataFrame(records)
    return df


def make_features(a, b, season, stats, form, seeds, elo_history, massey):
    """
    Difference-based features between team a and team b.
    """
    def get_stat(tid, col, default=0.0):
        try:
            return stats.at[tid, col]
        except:
            return default

    # ELO
    elo_a = elo_history.get((season, a), 1500)
    elo_b = elo_history.get((season, b), 1500)

    # Seeds
    seed_a = seeds.get(a, 16)
    seed_b = seeds.get(b, 16)

    # Massey rank (lower = better, so we flip sign for diff)
    m_a = massey.get(a, 200)
    m_b = massey.get(b, 200)

    feat = {
        "elo_diff"      : elo_a - elo_b,
        "seed_diff"     : seed_a - seed_b,   # negative = a is better seed
        "massey_diff"   : m_b - m_a,          # positive = a ranks better
        "win_pct_diff"  : get_stat(a, "win_pct") - get_stat(b, "win_pct"),
        "point_diff_diff": get_stat(a,"point_diff") - get_stat(b,"point_diff"),
        "avg_pts_diff"  : get_stat(a, "avg_pts") - get_stat(b, "avg_pts"),
        "avg_pts_ag_diff": get_stat(a,"avg_pts_ag") - get_stat(b,"avg_pts_ag"),
        "fg_pct_diff"   : get_stat(a,"fg_pct")  - get_stat(b,"fg_pct"),
        "fg3_pct_diff"  : get_stat(a,"fg3_pct") - get_stat(b,"fg3_pct"),
        "ft_pct_diff"   : get_stat(a,"ft_pct")  - get_stat(b,"ft_pct"),
        "avg_or_diff"   : get_stat(a,"avg_or")  - get_stat(b,"avg_or"),
        "avg_dr_diff"   : get_stat(a,"avg_dr")  - get_stat(b,"avg_dr"),
        "avg_ast_diff"  : get_stat(a,"avg_ast") - get_stat(b,"avg_ast"),
        "avg_to_diff"   : get_stat(a,"avg_to")  - get_stat(b,"avg_to"),
        "avg_stl_diff"  : get_stat(a,"avg_stl") - get_stat(b,"avg_stl"),
        "avg_blk_diff"  : get_stat(a,"avg_blk") - get_stat(b,"avg_blk"),
        "off_eff_diff"  : get_stat(a,"off_eff") - get_stat(b,"off_eff"),
        "def_eff_diff"  : get_stat(a,"def_eff") - get_stat(b,"def_eff"),
        "form_diff"     : form.get(a, 0.5) - form.get(b, 0.5),
    }
    return feat


# Build training data
print("  Building Men's training data …")
m_train = build_matchup_features(
    m_tourney, m_reg_compact, m_reg_detailed,
    m_seeds, m_elo_history, m_ordinal
)

print("  Building Women's training data …")
w_train = build_matchup_features(
    w_tourney, w_reg_compact, w_reg_detailed,
    w_seeds, w_elo_history
)

print(f"  M training samples : {len(m_train)}")
print(f"  W training samples : {len(w_train)}")

FEATURE_COLS = [c for c in m_train.columns if c != "label"]


# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Train LightGBM + XGBoost, tune ensemble weight
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("STEP 5 — Training LightGBM + XGBoost")
print("="*70)

# Best BO params (hand-tuned / typical winners for MM)
M_LGB_PARAMS = {
    "objective"       : "binary",
    "metric"          : "binary_logloss",
    "learning_rate"   : 0.03,
    "num_leaves"      : 31,
    "max_depth"       : 6,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq"    : 5,
    "reg_alpha"       : 0.1,
    "reg_lambda"      : 1.0,
    "n_estimators"    : 500,
    "random_state"    : 42,
    "verbose"         : -1,
}

W_LGB_PARAMS = {
    "objective"       : "binary",
    "metric"          : "binary_logloss",
    "learning_rate"   : 0.03,
    "num_leaves"      : 25,
    "max_depth"       : 5,
    "min_child_samples": 15,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq"    : 5,
    "reg_alpha"       : 0.05,
    "reg_lambda"      : 1.0,
    "n_estimators"    : 400,
    "random_state"    : 42,
    "verbose"         : -1,
}

M_XGB_PARAMS = {
    "objective"      : "binary:logistic",
    "eval_metric"    : "logloss",
    "learning_rate"  : 0.03,
    "max_depth"      : 5,
    "min_child_weight": 5,
    "subsample"      : 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha"      : 0.1,
    "reg_lambda"     : 1.0,
    "n_estimators"   : 500,
    "random_state"   : 42,
    "verbosity"      : 0,
}

W_XGB_PARAMS = {
    "objective"      : "binary:logistic",
    "eval_metric"    : "logloss",
    "learning_rate"  : 0.03,
    "max_depth"      : 4,
    "min_child_weight": 5,
    "subsample"      : 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha"      : 0.05,
    "reg_lambda"     : 1.0,
    "n_estimators"   : 400,
    "random_state"   : 42,
    "verbosity"      : 0,
}


def train_models(train_df, lgb_params, xgb_params, label=""):
    X = train_df[FEATURE_COLS].values
    y = train_df["label"].values

    models = {}

    if HAS_LGB:
        print(f"  [{label}] Training LightGBM …")
        clf_lgb = lgb.LGBMClassifier(**lgb_params)
        clf_lgb.fit(X, y)
        cv_lgb = cross_val_score(clf_lgb, X, y, cv=5,
                                  scoring="neg_log_loss").mean()
        print(f"         LGB CV log-loss : {-cv_lgb:.4f}")
        models["lgb"] = clf_lgb

    if HAS_XGB:
        print(f"  [{label}] Training XGBoost …")
        clf_xgb = xgb.XGBClassifier(**xgb_params)
        clf_xgb.fit(X, y)
        cv_xgb = cross_val_score(clf_xgb, X, y, cv=5,
                                  scoring="neg_log_loss").mean()
        print(f"         XGB CV log-loss : {-cv_xgb:.4f}")
        models["xgb"] = clf_xgb

    # Tune ensemble weight by CV log-loss on grid
    if HAS_LGB and HAS_XGB:
        best_w, best_loss = 0.5, 999
        for w in np.arange(0.0, 1.05, 0.05):
            blend = (w * models["lgb"].predict_proba(X)[:, 1] +
                     (1 - w) * models["xgb"].predict_proba(X)[:, 1])
            blend = np.clip(blend, 1e-7, 1 - 1e-7)
            loss = -np.mean(y * np.log(blend) + (1 - y) * np.log(1 - blend))
            if loss < best_loss:
                best_loss = loss
                best_w = w
        print(f"  [{label}] Best ensemble weight lgb={best_w:.2f} / xgb={1-best_w:.2f}")
        models["ensemble_w"] = best_w
    elif HAS_LGB:
        models["ensemble_w"] = 1.0
    else:
        models["ensemble_w"] = 0.0

    return models


m_models = train_models(m_train, M_LGB_PARAMS, M_XGB_PARAMS, label="MEN")
w_models = train_models(w_train, W_LGB_PARAMS, W_XGB_PARAMS, label="WOMEN")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — Build 2026 Prediction Features
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("STEP 6 — Building 2026 prediction features (2026 regular season)")
print("="*70)

PRED_SEASON = 2026

m_stats_2026 = build_team_stats(m_reg_detailed, m_reg_compact, PRED_SEASON)
w_stats_2026 = build_team_stats(w_reg_detailed, w_reg_compact, PRED_SEASON)

m_form_2026  = build_recent_form(m_reg_compact, PRED_SEASON)
w_form_2026  = build_recent_form(w_reg_compact, PRED_SEASON)

m_seeds_2026 = build_seed_dict(m_seeds, PRED_SEASON)
w_seeds_2026 = build_seed_dict(w_seeds, PRED_SEASON)

m_massey_2026 = build_massey(m_ordinal, PRED_SEASON)

print(f"  M teams with 2026 stats : {len(m_stats_2026)}")
print(f"  W teams with 2026 stats : {len(w_stats_2026)}")
print(f"  M teams seeded in 2026  : {len(m_seeds_2026)}")
print(f"  W teams seeded in 2026  : {len(w_seeds_2026)}")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — Predict all rows → clip to [0.025, 0.975]
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("STEP 7 — Predicting all submission rows")
print("="*70)

def predict_proba(models, feat_dict):
    X = np.array([[feat_dict.get(c, 0.0) for c in FEATURE_COLS]])
    w = models["ensemble_w"]
    if HAS_LGB and HAS_XGB:
        p_lgb = models["lgb"].predict_proba(X)[0, 1]
        p_xgb = models["xgb"].predict_proba(X)[0, 1]
        return w * p_lgb + (1 - w) * p_xgb
    elif HAS_LGB:
        return models["lgb"].predict_proba(X)[0, 1]
    else:
        return models["xgb"].predict_proba(X)[0, 1]


preds = []
skipped = 0

for _, row in sample_sub.iterrows():
    # Parse ID  e.g. "2026_1101_1102"
    parts  = row["ID"].split("_")
    season = int(parts[0])
    t_a    = int(parts[1])
    t_b    = int(parts[2])

    # Route to correct gender by TeamID prefix
    is_men = t_a < 2000 or t_b < 2000   # 1xxx = men, 3xxx = women

    if is_men:
        stats    = m_stats_2026
        form     = m_form_2026
        seeds    = m_seeds_2026
        elo_hist = m_elo_history
        massey   = m_massey_2026
        models   = m_models
    else:
        stats    = w_stats_2026
        form     = w_form_2026
        seeds    = w_seeds_2026
        elo_hist = w_elo_history
        massey   = {}
        models   = w_models

    feat = make_features(t_a, t_b, season, stats, form, seeds, elo_hist, massey)
    prob = predict_proba(models, feat)
    prob = float(np.clip(prob, 0.025, 0.975))

    preds.append({"ID": row["ID"], "Pred": prob})


# ─────────────────────────────────────────────────────────────────────────────
# STEP 8 — Save + Sanity Check
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("STEP 8 — Saving + sanity check")
print("="*70)

submission = pd.DataFrame(preds)

# Sanity checks
assert len(submission) == len(sample_sub), \
    f"Row count mismatch: {len(submission)} vs expected {len(sample_sub)}"
assert submission["Pred"].between(0.025, 0.975).all(), \
    "Some predictions outside [0.025, 0.975]!"
assert submission["ID"].equals(sample_sub["ID"]), \
    "ID column mismatch!"

print(f"  Rows            : {len(submission):,}")
print(f"  Pred min        : {submission.Pred.min():.4f}")
print(f"  Pred max        : {submission.Pred.max():.4f}")
print(f"  Pred mean       : {submission.Pred.mean():.4f}")
print(f"  Pred std        : {submission.Pred.std():.4f}")

submission.to_csv(OUT_NAME, index=False)

print(f"\n✅  Saved → {OUT_NAME}")
print("="*70)
print(submission.head(10).to_string(index=False))
print("="*70)


STEP 1 — Loading data
  Submission rows : 132,133
  M tourney games : 2,585
  W tourney games : 1,717
  M reg detailed  : 124,529  (seasons: 2003–2026)
  W reg detailed  : 87,187  (seasons: 2010–2026)

STEP 2 — Computing ELO ratings (2010–2026)
  M unique teams with ELO : 370
  W unique teams with ELO : 368

STEP 3 — Building team stats / recent form / Massey / seeds

STEP 4 — Building training matchup features
  Building Men's training data …
  Building Women's training data …
  M training samples : 1001
  W training samples : 961

STEP 5 — Training LightGBM + XGBoost
  [MEN] Training LightGBM …
         LGB CV log-loss : 0.6935
  [MEN] Training XGBoost …
         XGB CV log-loss : 0.6707
  [MEN] Best ensemble weight lgb=1.00 / xgb=0.00
  [WOMEN] Training LightGBM …
         LGB CV log-loss : 0.5266
  [WOMEN] Training XGBoost …
         XGB CV log-loss : 0.5070
  [WOMEN] Best ensemble weight lgb=1.00 / xgb=0.00

STEP 6 — Building 2026 prediction features (2026 regular season)
  M tea